# 🇬🇭 Twi Dataset Generation Pipeline

> Created with ❤️ by **Mich-Seth Owusu** for the Ghana NLP Community

This notebook walks through the full pipeline for generating a large-scale Twi language dataset:

1. **Setup** — install dependencies, configure API keys
2. **Generation** — use Gemini API to generate Twi text from English news paragraphs
3. **Cleaning** — remove speaker labels, markdown artifacts, validate text
4. **Push to HuggingFace** — upload as a clean columnar Parquet dataset with a README

---

### Prerequisites
- A [Gemini API key](https://aistudio.google.com/app/apikey)
- A [HuggingFace account](https://huggingface.co) with a write token
- Access to the source dataset: `ghananlpcommunity/twi-english-paragraph-dataset_news`


---
## 📦 Section 1 — Install Dependencies

In [ ]:
!pip install -q google-generativeai datasets huggingface_hub pandas tqdm

---
## 🔑 Section 2 — Configuration

Fill in your credentials and settings below before running the pipeline.

In [ ]:
# ════════════════════════════════════════════════════════════
#  FILL IN YOUR CREDENTIALS HERE
# ════════════════════════════════════════════════════════════

GEMINI_API_KEY = "YOUR_GEMINI_API_KEY_HERE"   # https://aistudio.google.com/app/apikey
HF_TOKEN       = "YOUR_HF_TOKEN_HERE"         # https://huggingface.co/settings/tokens
HF_REPO_ID     = "your-username/your-dataset-name"  # e.g. ghananlpcommunity/pristine-twi

# ════════════════════════════════════════════════════════════
#  GENERATION SETTINGS
# ════════════════════════════════════════════════════════════

GEMINI_MODEL        = "gemini-2.0-flash-lite"
MAX_CONCURRENT      = 15       # parallel workers — increase if you have a higher quota
REQUESTS_PER_MINUTE = 60       # set to ~80% of your actual Gemini RPM quota
START_INDEX         = 0        # which row to start from in the source dataset
END_INDEX           = 200000   # how many rows to process (capped by dataset size)
OUTPUT_FILE         = "twi_texts.jsonl"  # where generated text is saved
PRIVATE_REPO        = False    # set True to make your HF dataset private

# Derived
RATE_LIMIT_DELAY = 60 / REQUESTS_PER_MINUTE

print("✓ Config loaded")
print(f"  Model      : {GEMINI_MODEL}")
print(f"  Workers    : {MAX_CONCURRENT}")
print(f"  RPM        : {REQUESTS_PER_MINUTE}")
print(f"  Row range  : {START_INDEX} → {END_INDEX}")
print(f"  Output     : {OUTPUT_FILE}")
print(f"  HF repo    : {HF_REPO_ID}")

---
## 📂 Section 3 — Load Source Dataset

In [ ]:
from datasets import load_dataset
import pandas as pd

DATASET_REPO = "ghananlpcommunity/twi-english-paragraph-dataset_news"

print(f"Loading dataset: {DATASET_REPO}")
ds = load_dataset(DATASET_REPO, split="train", trust_remote_code=True)
df = ds.to_pandas()

end = min(END_INDEX, len(df))
paragraphs = df.iloc[START_INDEX:end]["ENGLISH"].dropna().tolist()

print(f"✓ Loaded {len(paragraphs):,} paragraphs (rows {START_INDEX}–{end-1})")
print(f"\nSample paragraph:")
print(paragraphs[0][:300], "...")

---
## ✍️ Section 4 — Twi Text Generation

Generates ~2,000 words of Twi per row across 4 styles:
- **Monologue** (*ɔkasa biako*) — passionate, personal
- **Narrative** (*abakɔsɛm*) — journalistic, structured
- **Dialogue** (*nkɔmmɔdie*) — conversational
- **Story** (*anansesɛm*) — dramatic, metaphorical

> ⚡ Rows are **never skipped** — the generator retries indefinitely with exponential backoff on any error (rate limits, network drops, validation failures).
>
> 💾 Progress is saved after every single row — safe to stop and resume at any time.

In [ ]:
import asyncio
import json
import re
import time
from pathlib import Path
from datetime import datetime

# ── Prompts ───────────────────────────────────────────────────────────────────

PROMPT_TEMPLATE_1 = """Generate Twi text in 4 parts (each approximately 500 words):
1. Monologue (ɔkasa biako) - passionate, personal perspective
2. Narrative (abakɔsɛm) - journalistic, structured account
3. Dialogue (nkɔmmɔdie) - conversational debate between 2 people
4. Storyful (anansesɛm) - dramatic, metaphorical storytelling

Use this English paragraph as inspiration:
{paragraph}

Guidelines:
- Mix English words naturally as done in spoken Twi
- Write in natural, flowing Twi
- Each part should be substantial (~500 words)
- Return ONLY the Twi text, no preamble, no markdown formatting, no section headers"""

PROMPT_TEMPLATE_2 = """Create comprehensive Twi text with 4 distinct sections (each ~500 words):
- A passionate monologue
- A journalistic narrative
- A conversational dialogue
- A dramatic story

Based on this topic:
{paragraph}

Important:
- Write naturally in Twi, mixing English words where appropriate
- Make each section substantial and complete
- Total output should be around 2000 words
- Return plain Twi text only, no formatting or labels"""

# ── Validation ────────────────────────────────────────────────────────────────

MIN_CHARS = 2000
MAX_CHARS = 18000

def remove_consecutive_repetitions(text):
    parts = re.split(r'(\s*[.!?\n]+\s*)', text)
    sentences = []
    for i in range(0, len(parts) - 1, 2):
        s = parts[i].strip()
        delim = parts[i+1] if i+1 < len(parts) else " "
        if s:
            sentences.append((s, delim))
    if parts and parts[-1].strip():
        sentences.append((parts[-1].strip(), ""))
    if not sentences:
        return text, 0
    cleaned = [sentences[0]]
    removed = 0
    i = 1
    while i < len(sentences):
        matched = False
        max_block = (len(sentences) - i) // 2
        for k in range(min(max_block, 10), 0, -1):
            block = [s for s, _ in sentences[i:i+k]]
            prev_block = [s for s, _ in cleaned[-k:]] if len(cleaned) >= k else None
            if prev_block and [s.lower() for s in block] == [s.lower() for s in prev_block]:
                removed += k
                i += k
                matched = True
                break
        if not matched:
            cleaned.append(sentences[i])
            i += 1
    return " ".join(s + d for s, d in cleaned).strip(), removed

def validate_text(text):
    if not text or not text.strip():
        return "", 0, 0, False, "Empty response"
    cleaned, n_removed = remove_consecutive_repetitions(text.strip())
    char_count = len(cleaned)
    if char_count < MIN_CHARS:
        return cleaned, char_count, n_removed, False, f"Too short: {char_count:,} chars"
    if char_count > MAX_CHARS:
        return cleaned, char_count, n_removed, False, f"Too long: {char_count:,} chars"
    return cleaned, char_count, n_removed, True, None

print("✓ Generation helpers defined")

In [ ]:
# ── Generator class ───────────────────────────────────────────────────────────

INITIAL_BACKOFF  = 5
MAX_BACKOFF      = 300
BACKOFF_FACTOR   = 2
RATE_LIMIT_PAUSE = 60

class GeminiGenerator:
    def __init__(self, api_key, output_file=OUTPUT_FILE):
        self.api_key            = api_key
        self.output_file        = Path(output_file)
        self.completed_indices  = set()
        self.semaphore          = asyncio.Semaphore(MAX_CONCURRENT)
        self.rate_limiter       = asyncio.Lock()
        self.last_request_time  = 0.0
        self._file_lock         = asyncio.Lock()
        self._load_progress()

    def _load_progress(self):
        if self.output_file.exists():
            with open(self.output_file, 'r', encoding='utf-8') as f:
                for line in f:
                    try:
                        self.completed_indices.add(json.loads(line.strip())['index'])
                    except Exception:
                        pass
        print(f"  Resuming: {len(self.completed_indices):,} rows already done.")

    async def _rate_limit(self):
        async with self.rate_limiter:
            wait = RATE_LIMIT_DELAY - (time.monotonic() - self.last_request_time)
            if wait > 0:
                await asyncio.sleep(wait)
            self.last_request_time = time.monotonic()

    async def _call_gemini(self, prompt):
        import google.generativeai as genai
        await self._rate_limit()
        genai.configure(api_key=self.api_key)
        model = genai.GenerativeModel(GEMINI_MODEL)
        response = await asyncio.to_thread(
            model.generate_content, prompt,
            generation_config={'temperature': 1.0, 'top_p': 0.95,
                               'top_k': 64, 'max_output_tokens': 8192}
        )
        return response.text if response.text else None

    async def _generate_forever(self, index, paragraph):
        attempt, backoff, prompt_idx = 0, INITIAL_BACKOFF, 0
        templates = [PROMPT_TEMPLATE_1, PROMPT_TEMPLATE_2]
        while True:
            attempt += 1
            prompt = templates[prompt_idx % 2].format(paragraph=paragraph)
            prompt_idx += 1
            try:
                response_text = await self._call_gemini(prompt)
            except Exception as e:
                err = str(e)
                if "429" in err or "quota" in err.lower():
                    wait = RATE_LIMIT_PAUSE + backoff
                    print(f"  [{index:06d}] Rate-limited — sleeping {wait}s")
                    await asyncio.sleep(wait)
                else:
                    print(f"  [{index:06d}] Error (attempt {attempt}): {e} — retry in {backoff}s")
                    await asyncio.sleep(backoff)
                    backoff = min(backoff * BACKOFF_FACTOR, MAX_BACKOFF)
                continue
            if response_text is None:
                print(f"  [{index:06d}] Empty response — retry in {backoff}s")
                await asyncio.sleep(backoff)
                backoff = min(backoff * BACKOFF_FACTOR, MAX_BACKOFF)
                continue
            cleaned, char_count, reps_removed, is_valid, error = validate_text(response_text)
            if is_valid:
                return {
                    'index': index, 'twi_text': cleaned,
                    'char_count': char_count, 'repetitions_removed': reps_removed,
                    'attempts': attempt, 'model': GEMINI_MODEL,
                    'timestamp': datetime.now().isoformat()
                }
            print(f"  [{index:06d}] Validation failed: {error} — retry in {backoff}s")
            await asyncio.sleep(backoff)
            backoff = min(backoff * BACKOFF_FACTOR, MAX_BACKOFF)

    async def _process_one(self, index, paragraph, counter):
        if index in self.completed_indices:
            counter['skipped'] += 1
            return
        async with self.semaphore:
            result = await self._generate_forever(index, paragraph)
            async with self._file_lock:
                with open(self.output_file, 'a', encoding='utf-8') as f:
                    f.write(json.dumps(result, ensure_ascii=False) + '\n')
                self.completed_indices.add(index)
            counter['done'] += 1
            print(f"  [{index:06d}] ✓ saved ({result['char_count']:,} chars) | total: {len(self.completed_indices):,}")

    async def _progress_ticker(self, counter, total):
        start = time.monotonic()
        try:
            while True:
                await asyncio.sleep(30)
                elapsed = time.monotonic() - start
                done    = len(self.completed_indices)
                rate    = done / (elapsed / 60) if elapsed > 0 else 0
                eta     = (total - done) / rate if rate > 0 else float('inf')
                print(f"\n  ── Progress: {done:,}/{total:,} | {rate:.1f} rows/min | ETA ≈ {eta:.0f} min ──\n")
        except asyncio.CancelledError:
            pass

    async def generate_batch(self, paragraphs, start_index=0):
        pending = [
            (start_index + i, para)
            for i, para in enumerate(paragraphs)
            if (start_index + i) not in self.completed_indices
        ]
        print(f"\n{'='*60}")
        print(f"  Total rows       : {len(paragraphs):,}")
        print(f"  Already done     : {len(paragraphs) - len(pending):,}")
        print(f"  To process       : {len(pending):,}")
        print(f"  Model            : {GEMINI_MODEL}")
        print(f"  Workers          : {MAX_CONCURRENT}")
        print(f"  RPM              : {REQUESTS_PER_MINUTE}")
        print(f"  Never skips rows : True")
        print(f"{'='*60}\n")
        counter = {'done': 0, 'skipped': 0}
        ticker  = asyncio.create_task(self._progress_ticker(counter, len(paragraphs)))
        await asyncio.gather(*[self._process_one(idx, para, counter) for idx, para in pending])
        ticker.cancel()
        print(f"\n✓ Batch complete — {len(self.completed_indices):,} rows saved to {self.output_file}")

print("✓ GeminiGenerator class defined")

In [ ]:
# ── Run generation ────────────────────────────────────────────────────────────
# This cell runs the full generation pipeline.
# It is safe to interrupt (Kernel > Interrupt) and re-run — it will resume.

import nest_asyncio
nest_asyncio.apply()   # needed to run asyncio inside Jupyter

generator = GeminiGenerator(GEMINI_API_KEY, output_file=OUTPUT_FILE)
asyncio.run(generator.generate_batch(paragraphs, start_index=START_INDEX))

---
## 🧹 Section 5 — Clean the Generated Text

Applies the following cleaning steps:
- **Remove speaker labels** — `Kofi:`, `Yaw:`, `Ama:` etc.
- **Strip markdown** — `**bold**`, `## headers`
- **Normalize unicode** — zero-width chars, non-breaking spaces
- **Collapse whitespace** — excessive blank lines, tab runs
- **Validate** — drop rows that are too short or look like English was returned

In [ ]:
import json
import re
from pathlib import Path

# ── Speaker label pattern ─────────────────────────────────────────────────────
_SPEAKER_RE = re.compile(
    r'(?m)^\s{0,10}[A-Z][A-Za-z]+(?:\s+[A-Z][A-Za-z]+)?\s*:\s*'
)

def clean_twi_text(text):
    if not text or not isinstance(text, str):
        return ""
    text = _SPEAKER_RE.sub('', text)
    text = re.sub(r'\*{1,3}(.*?)\*{1,3}', r'\1', text, flags=re.DOTALL)
    text = re.sub(r'^#{1,6}\s+', '', text, flags=re.MULTILINE)
    text = text.replace('\u200b', '').replace('\u00a0', ' ').replace('\ufeff', '')
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]{2,}', ' ', text)
    return text.strip()

def is_valid_twi(twi):
    if not twi or len(twi) < 500:
        return False, f"Too short: {len(twi)} chars"
    ascii_ratio = sum(1 for c in twi if ord(c) < 128) / len(twi)
    if ascii_ratio > 0.98:
        return False, f"Looks like English (ascii ratio {ascii_ratio:.2f})"
    return True, ""

def load_and_clean(jsonl_path):
    records, errors = [], 0
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"  Warning line {i+1}: {e}")
                errors += 1
    print(f"Loaded {len(records):,} records ({errors} parse errors)")

    rows  = []
    stats = {'total': len(records), 'valid': 0, 'dropped': 0, 'duplicates': 0}
    seen  = set()

    for rec in records:
        idx = rec.get('index')
        if idx in seen:
            stats['duplicates'] += 1
            continue
        seen.add(idx)

        twi = clean_twi_text(rec.get('twi_text', ''))
        valid, reason = is_valid_twi(twi)
        if not valid:
            print(f"  Drop [{idx}]: {reason}")
            stats['dropped'] += 1
            continue

        rows.append({'twi': twi, 'char_count': len(twi)})
        stats['valid'] += 1

    return rows, stats

print("✓ Cleaning helpers defined")

In [ ]:
# ── Preview cleaning on first 3 rows ─────────────────────────────────────────
preview_records = []
with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        if len(preview_records) >= 3:
            break
        try:
            preview_records.append(json.loads(line.strip()))
        except Exception:
            pass

print("── CLEANING PREVIEW ──")
for rec in preview_records:
    raw   = rec.get('twi_text', '')
    clean = clean_twi_text(raw)
    print(f"\nIndex  : {rec.get('index')}")
    print(f"Before : {raw[:120]!r}")
    print(f"After  : {clean[:120]!r}")
    print(f"Chars  : {len(raw):,} → {len(clean):,}")

In [ ]:
# ── Run full cleaning ─────────────────────────────────────────────────────────
clean_rows, stats = load_and_clean(OUTPUT_FILE)

print(f"\n── Cleaning Summary ──")
print(f"  Total loaded : {stats['total']:,}")
print(f"  Valid        : {stats['valid']:,}")
print(f"  Duplicates   : {stats['duplicates']:,}")
print(f"  Dropped      : {stats['dropped']:,}")

# Preview final cleaned rows
import pandas as pd
df_clean = pd.DataFrame(clean_rows)[['twi', 'char_count']]
print(f"\nDataFrame shape: {df_clean.shape}")
df_clean.head(3)

---
## 🚀 Section 6 — Push to HuggingFace

Pushes the cleaned dataset as **Parquet** (columnar format) with a README.

Final columns: `twi` | `char_count`

In [ ]:
from datetime import datetime

README = f"""---
language:
- tw
license: cc-by-4.0
task_categories:
- text-generation
- fill-mask
pretty_name: Pristine Twi
tags:
- twi
- akan
- ghana
- nlp
- africa
---

# Pristine Twi Dataset

> Created with ❤️ by **Mich-Seth Owusu** for the Ghana NLP Community

## Overview

A large-scale Twi language dataset containing naturally written Twi text across
four distinct styles — monologue, narrative, dialogue, and storytelling — generated
from real Ghanaian news topics as inspiration.

This dataset was built to support the development of Twi language models, tokenizers,
and other NLP tools for the Akan language family.

## Dataset Details

| Field        | Detail                        |
|------------- |-------------------------------|
| Language     | Twi (tw) — Akan, Ghana        |
| License      | CC BY 4.0                     |
| Author       | Mich-Seth Owusu               |
| Organization | Ghana NLP Community           |
| Format       | Parquet (columnar)            |
| Rows         | {len(df_clean):,}             |

## Columns

| Column       | Type   | Description                              |
|--------------|--------|------------------------------------------|
| `twi`        | string | Twi text (~2,000 words per row)          |
| `char_count` | int    | Character count of the Twi text          |

## Text Styles

Each row contains Twi text written in four blended styles:

- **Monologue** (*ɔkasa biako*) — passionate, personal perspective
- **Narrative** (*abakɔsɛm*) — journalistic, structured account
- **Dialogue** (*nkɔmmɔdie*) — conversational exchange
- **Story** (*anansesɛm*) — dramatic, metaphorical storytelling

## Usage

```python
from datasets import load_dataset

ds = load_dataset("{HF_REPO_ID}", split="train")
print(ds[0]["twi"])
```

## Generation

Text was generated using the Gemini API with Ghanaian news paragraphs as
prompts, then cleaned to remove speaker labels, markdown artifacts, and
any responses that failed Twi language validation.

## Citation

```
@dataset{{pristine_twi_2026,
  author    = {{Mich-Seth Owusu}},
  title     = {{Pristine Twi Dataset}},
  year      = {{{datetime.now().year}}},
  publisher = {{HuggingFace}},
  url       = {{https://huggingface.co/datasets/{HF_REPO_ID}}}
}}
```

---

*Made with ❤️ by Mich-Seth Owusu for the Ghana NLP Community*
"""

print("✓ README generated")
print(f"  Length: {len(README):,} chars")

In [ ]:
from datasets import Dataset
from huggingface_hub import HfApi

# ── Push dataset ──────────────────────────────────────────────────────────────
print(f"Pushing dataset to: {HF_REPO_ID}")
print(f"  Rows    : {len(df_clean):,}")
print(f"  Columns : {list(df_clean.columns)}")
print(f"  Private : {PRIVATE_REPO}")

hf_dataset = Dataset.from_pandas(df_clean, preserve_index=False)
hf_dataset.push_to_hub(
    HF_REPO_ID,
    private=PRIVATE_REPO,
    token=HF_TOKEN,
    commit_message=f"Add {len(df_clean):,} Twi texts — {datetime.now().strftime('%Y-%m-%d')}",
)
print("✓ Dataset pushed")

# ── Push README ───────────────────────────────────────────────────────────────
print("Pushing README...")
api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj=README.encode('utf-8'),
    path_in_repo="README.md",
    repo_id=HF_REPO_ID,
    repo_type="dataset",
    commit_message="Add dataset README",
)
print("✓ README pushed")
print(f"\n🎉 Done! View at: https://huggingface.co/datasets/{HF_REPO_ID}")

---
## ✅ Section 7 — Verify

Load the dataset back from HuggingFace to confirm everything looks correct.

In [ ]:
from datasets import load_dataset

print(f"Loading back from HF: {HF_REPO_ID}")
ds_verify = load_dataset(HF_REPO_ID, split="train", token=HF_TOKEN)

print(f"\n── Dataset Info ──")
print(f"  Rows    : {len(ds_verify):,}")
print(f"  Columns : {ds_verify.column_names}")
print(f"\n── Sample Row ──")
print(f"  char_count : {ds_verify[0]['char_count']:,}")
print(f"  twi        : {ds_verify[0]['twi'][:200]}...")

---

*Made with ❤️ by **Mich-Seth Owusu** for the Ghana NLP Community*